In [1]:
from pathlib import Path

from IPython.display import display

from searchlight.lightcurves import load
from searchlight.review_exports import (
    DEFAULT_METADATA_CACHE_PATH,
    DEFAULT_RESULT_CACHE_META_PATH,
    DEFAULT_RESULT_CACHE_PATH,
    enrich_results_with_metadata,
    load_lensing_results,
)
from searchlight.review_widgets import make_zooniverse_review_item
from searchlight.zooniverse_results import (
    DEFAULT_SUBJECT_METADATA_CACHE_PATH,
    add_candidate_ids,
    select_zooniverse_subject_results,
)


In [2]:
RESULTS_PATH = Path('data/results/simulens1.0_tess_spoc')
RESULT_CACHE_PATH = DEFAULT_RESULT_CACHE_PATH
RESULT_CACHE_META_PATH = DEFAULT_RESULT_CACHE_META_PATH
RESULT_METADATA_CACHE_PATH = DEFAULT_METADATA_CACHE_PATH

SUBJECT_METADATA_CACHE_PATH = DEFAULT_SUBJECT_METADATA_CACHE_PATH

# Add Zooniverse subject IDs or candidate UUIDs here.
# Numeric IDs are looked up through Zooniverse; UUIDs are matched directly.
REVIEW_IDS = ["33aa95b6-a354-5a83-ae13-92a53b9e2645", "1395e328-71da-550a-a746-64d303a1ee0e"]
TOP_N = None

# Leave these as None to use the automatic peak-centered zoom window.
# Use relative days from the first cadence, for example (12.4, 13.1).
ZOOM_WINDOW_DAYS = None
# Or use absolute JD timestamps, for example (2459800.4, 2459801.1).
ZOOM_WINDOW_JD = None

# Optional per-subject overrides when reviewing several subjects.
# Example: {119539666: (12.4, 13.1)}
ZOOM_WINDOWS_DAYS_BY_SUBJECT_ID = {}
ZOOM_WINDOWS_JD_BY_SUBJECT_ID = {}


In [3]:
# Subject metadata mapping and review-card rendering live in searchlight modules
# so this notebook and the CSV-driven notebook share the same implementation.


In [4]:
# Review widget implementation: searchlight.review_widgets.make_zooniverse_review_item


In [5]:
results = load_lensing_results(
    results_path=RESULTS_PATH,
    cache_path=RESULT_CACHE_PATH,
    cache_meta_path=RESULT_CACHE_META_PATH,
)
results = add_candidate_ids(results)
print(f'Loaded {len(results)} lensing model rows from {RESULTS_PATH}')


Loading cached results from data/results/simulens1.0_tess_spoc/top_resnet_results_cache.pkl
Loaded 2160368 lensing model rows from data/results/simulens1.0_tess_spoc


In [6]:
selected = select_zooniverse_subject_results(
    results,
    REVIEW_IDS,
    subject_metadata_cache_path=SUBJECT_METADATA_CACHE_PATH,
)
review_results = selected.copy()
if TOP_N is not None:
    review_results = review_results.head(TOP_N).copy()
review_results[['subject_id', 'candidate_id', 'path', 'lensing']].head()


Mapped 2 review IDs to local paths


,subject_id,candidate_id,path,lensing
0,None,33aa95b6-a354-5a83-ae13-92a53b9e2645,https://mast.stsci.edu/api/v0.1/Download/file/...,3.805800
1,None,1395e328-71da-550a-a746-64d303a1ee0e,https://mast.stsci.edu/api/v0.1/Download/file/...,4.509336


In [7]:
review_export = enrich_results_with_metadata(
    review_results,
    cache_path=RESULT_METADATA_CACHE_PATH,
)
review_export[[
    'subject_id',
    'candidate_id',
    'path',
    'lensing',
    'tic',
    'sector',
    'simbad_url',
]].head()


,subject_id,candidate_id,path,lensing,tic,sector,simbad_url
0,None,33aa95b6-a354-5a83-ae13-92a53b9e2645,https://mast.stsci.edu/api/v0.1/Download/file/...,3.805800,356770527,37,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
1,None,1395e328-71da-550a-a746-64d303a1ee0e,https://mast.stsci.edu/api/v0.1/Download/file/...,4.509336,380887437,70,https://simbad.cds.unistra.fr/simbad/sim-coo?C...


In [8]:
def _subject_zoom_window(mapping, subject_id):
    return mapping.get(subject_id, mapping.get(str(subject_id)))


for row in review_results.to_dict('records'):
    subject_id = row.get('subject_id')
    zoom_window_days = _subject_zoom_window(
        ZOOM_WINDOWS_DAYS_BY_SUBJECT_ID,
        subject_id,
    ) or ZOOM_WINDOW_DAYS
    zoom_window_jd = _subject_zoom_window(
        ZOOM_WINDOWS_JD_BY_SUBJECT_ID,
        subject_id,
    ) or ZOOM_WINDOW_JD
    lc = load(row['path'])
    display(
        make_zooniverse_review_item(
            row,
            lc,
            zoom_window_days=zoom_window_days,
            zoom_window_jd=zoom_window_jd,
        )
    )


0% (16/3477) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (16/3477) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).


2% (226/10730) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:2% (226/10730) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
